# Sesión 08 — Lab 2: AUTO CDC INTO con SCD Type 2
## Captura y propagación de cambios sin código manual

**Curso:** Databricks Data Engineer Associate
**Dataset:** `cambios_clientes.csv`

---

### Objetivos del laboratorio
1. Comprender cómo `dp.create_auto_cdc_flow` reemplaza al MERGE manual de la S05
2. Aplicar SCD Type 2 con manejo de DELETEs en eventos CDC
3. Consultar el estado actual y la historia completa de la dimensión

> **Recordatorio:** SDP usa `dp.create_auto_cdc_flow` (API moderna). El legacy `dlt.apply_changes` ya no se usa.

## Paso 1 — Subir el dataset al Volume

```
/Volumes/dbassociate/default/vol_landing/sesion_08/clientes/cambios_clientes.csv
```

Estructura: `evento_id, cliente_id, nombre, email, plan, status, operation, event_ts`
- `operation` ∈ {INSERT, UPDATE, DELETE}
- 100 eventos sobre 35 clientes únicos
- `event_ts` define el orden temporal (sequence_by)

## Paso 2 — Crear el Pipeline en la UI

1. Workspace → New → ETL Pipeline
2. **Pipeline name:** `dbassociate_s08_lab2`
3. **Source code:** este notebook
4. **Default schema:** `silver` (las tablas Bronze usan full qualified name)
5. **Compute:** Serverless
6. **Mode:** Triggered, Channel: Current

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

## Paso 3 — Bronze: ingesta de eventos CDC

Streaming Table que recibe los eventos crudos. Cada fila es un evento (INSERT/UPDATE/DELETE) con su timestamp.

In [0]:
@dp.table(
    name="dbassociate.bronze.cambios_clientes",
    comment="Eventos CDC de cambios en clientes (INSERT/UPDATE/DELETE).",
    table_properties={"quality": "bronze"},
)
def bronze_cambios_clientes():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.schemaLocation",
                    "/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_cambios_clientes")
            .load("/Volumes/dbassociate/default/vol_landing/sesion_08/clientes/")
            .withColumn("event_ts", F.to_timestamp("event_ts"))
            .withColumn("_ingested_at", F.current_timestamp())
    )

## Paso 4 — Silver: dim_clientes con SCD Type 2

Patrón en dos pasos:

1. **`dp.create_streaming_table`** declara la tabla destino (vacía).
2. **`dp.create_auto_cdc_flow`** define cómo se alimenta desde la fuente CDC.

Parámetros clave:
- `keys=["cliente_id"]` — clave de negocio
- `sequence_by=F.col("event_ts")` — orden temporal de los eventos
- `apply_as_deletes=F.expr("operation = 'DELETE'")` — qué eventos representan eliminaciones (en SCD2 cierran el `__END_AT`)
- `except_column_list=[...]` — columnas a excluir del destino (metadata CDC)
- `stored_as_scd_type=2` — mantener historia con `__START_AT` / `__END_AT`

In [0]:
dp.create_streaming_table(
    name="dbassociate.silver.dim_clientes",
    comment="Dimensión de clientes con historia completa (SCD Type 2).",
    table_properties={"quality": "silver"},
)

dp.create_auto_cdc_flow(
    target="dbassociate.silver.dim_clientes",
    source="dbassociate.bronze.cambios_clientes",
    keys=["cliente_id"],
    sequence_by=F.col("event_ts"),
    apply_as_deletes=F.expr("operation = 'DELETE'"),
    except_column_list=["evento_id", "operation", "event_ts", "_ingested_at"],
    stored_as_scd_type=2,
)

## Paso 5 — Ejecutar y verificar

Run el pipeline. Después, en SQL editor:

```sql
-- ESTADO ACTUAL: filas vigentes (no eliminadas) tienen __END_AT IS NULL
SELECT cliente_id, nombre, email, plan, status, __START_AT
FROM dbassociate.silver.dim_clientes
WHERE __END_AT IS NULL
ORDER BY cliente_id;

-- HISTORIA de un cliente que tuvo varios cambios
SELECT cliente_id, nombre, email, plan, status, __START_AT, __END_AT
FROM dbassociate.silver.dim_clientes
WHERE cliente_id = 'CLI-001'
ORDER BY __START_AT;

-- Clientes eliminados (DELETE event): aparecen con __END_AT no nulo y sin fila vigente posterior
SELECT cliente_id, MAX(__END_AT) AS deleted_at
FROM dbassociate.silver.dim_clientes
GROUP BY cliente_id
HAVING SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END) = 0;
```

**Resultado esperado:**
- `dim_clientes` tiene más filas que clientes únicos (porque hay historia)
- Clientes con DELETE: todas sus filas tienen `__END_AT` no nulo
- Clientes con UPDATE: filas históricas con `__END_AT` no nulo + 1 fila vigente con `__END_AT IS NULL`

## Antipatrones marcados

- `dlt.apply_changes(...)` legacy → `dp.create_auto_cdc_flow(...)`
- Escribir MERGE INTO manual con `WHEN MATCHED UPDATE SET __END_AT = current_timestamp()` → `AUTO CDC INTO` lo hace por ti
- Filtrar columnas con `START_AT` / `END_AT` (sin doble underscore) → la columna real es `__START_AT` y `__END_AT`
- Olvidar `apply_as_deletes` cuando hay DELETEs → SDP no sabe cómo cerrar las filas eliminadas
- Usar `sequence_by` con una columna que no es monotónica → orden incorrecto de los cambios

In [0]:
# LIMPIEZA — ejecutar en SQL editor
#
# DROP TABLE IF EXISTS dbassociate.bronze.cambios_clientes;
# DROP TABLE IF EXISTS dbassociate.silver.dim_clientes;
# -- Borrar el pipeline desde la UI
# dbutils.fs.rm("/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_cambios_clientes", recurse=True)